# Exploratory Data Analysis: Raw EPA Data



In [14]:
from __future__ import annotations

import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

print("Libraries imported successfully.")

Libraries imported successfully.


## Run the stage


Enhanced exploratory data analysis for the raw EPA annual air-quality summary CSV.

Covers:
- Dataset structure and variable types
- Missing-value analysis (counts, percentages, patterns, visualisation)
- Duplicate detection
- Outlier analysis (IQR method + boxplots)
- Class imbalance (for proposed classification target)
- Distributions by pollutant, state, and year
- Correlation analysis (numeric features)
- Target-variable proposal and analysis
- Data-leakage risk identification

Outputs:
- reports/raw_eda/*.csv
- reports/raw_eda/*.png
- reports/raw_eda/raw_eda_report.md



Configuration

In [15]:
# ------------------------------------------------------------------
# Resolve the raw CSV path automatically from the current working dir
# ------------------------------------------------------------------
# The notebook is expected to run from notebooks/.
# If Path.cwd() is the project root, adjust accordingly.

_cwd = Path.cwd()

# Case 1: running from notebooks/ (VS Code default)
if (_cwd.parent / "data" / "raw").exists():
    PROJECT_ROOT = _cwd.parent

# Case 2: running from the project root itself
elif (_cwd / "data" / "raw").exists():
    PROJECT_ROOT = _cwd

# Case 3: running from somewhere deeper (e.g. notebooks/subfolder/)
else:
    # Walk up until we find the data/raw folder
    PROJECT_ROOT = None
    for candidate in [_cwd, *_cwd.parents]:
        if (candidate / "data" / "raw").exists():
            PROJECT_ROOT = candidate
            break

    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            f"Could not locate the project root starting from {_cwd}.\n"
            "Expected to find a 'data/raw/' folder somewhere up the tree.\n"
            "Make sure 'air_quality.csv' exists at <project>/data/raw/air_quality.csv"
        )

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "air_quality.csv"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "raw_eda"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw CSV path: {RAW_PATH}")
print(f"CSV exists  : {RAW_PATH.exists()}")

Project root: g:\ACHINI\sliit activities\Y3S2\Data Science\FDM\FDM-Mini-Project
Raw CSV path: g:\ACHINI\sliit activities\Y3S2\Data Science\FDM\FDM-Mini-Project\data\raw\air_quality.csv
CSV exists  : True


Define Columns

In [16]:
NUMERIC_COLUMNS = [
    "latitude",
    "longitude",
    "year",
    "observation_count",
    "observation_percent",
    "valid_day_count",
    "arithmetic_mean",
    "arithmetic_standard_dev",
    "first_max_value",
    "ninety_five_percentile",
    "fifty_percentile",
]

CATEGORY_COLUMNS = [
    "parameter_name",
    "units_of_measure",
    "sample_duration",
    "state_name",
]

EXTRA_COLUMNS = [
    "state_code",
    "county_code",
    "site_number",
    "datum",
]

print("Column lists defined.")

Column lists defined.


Helper Functions

In [17]:
def ensure_output_dir():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(path):
    """
    Save the figure and display it inside Jupyter Notebook.
    """
    plt.tight_layout()

    plt.savefig(
        path,
        dpi=160,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


def print_section(title):
    print("\n")
    print("=" * 80)
    print(title)
    print("=" * 80)


print("Helper functions created.")

Helper functions created.


Check CSV File

In [18]:
print_section("AIR QUALITY RAW DATA EDA")

print("\nCSV path:")
print(RAW_PATH)

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"""
CSV file not found!

Expected path:
{RAW_PATH}
"""
    )

print("\n✓ CSV file found successfully!")

ensure_output_dir()

print("\nOutput directory:")
print(OUTPUT_DIR)



AIR QUALITY RAW DATA EDA

CSV path:
g:\ACHINI\sliit activities\Y3S2\Data Science\FDM\FDM-Mini-Project\data\raw\air_quality.csv

✓ CSV file found successfully!

Output directory:
g:\ACHINI\sliit activities\Y3S2\Data Science\FDM\FDM-Mini-Project\reports\raw_eda


Read CSV Header

In [19]:
print_section("STEP 1 - READING CSV HEADER")

header = pd.read_csv(
    RAW_PATH,
    nrows=0
)

all_columns = list(header.columns)

print(f"\nTotal columns in CSV: {len(all_columns)}")

print("\nFirst 20 columns:")

for col in all_columns[:20]:
    print(" -", col)



STEP 1 - READING CSV HEADER

Total columns in CSV: 55

First 20 columns:
 - state_code
 - county_code
 - site_num
 - parameter_code
 - poc
 - latitude
 - longitude
 - datum
 - parameter_name
 - sample_duration
 - pollutant_standard
 - metric_used
 - method_name
 - year
 - units_of_measure
 - event_type
 - observation_count
 - observation_percent
 - completeness_indicator
 - valid_day_count


Select Available Columns

In [20]:
print_section("STEP 2 - SELECTING COLUMNS")

requested_columns = sorted(
    set(
        NUMERIC_COLUMNS
        + CATEGORY_COLUMNS
        + EXTRA_COLUMNS
    )
)

USECOLS = [
    col
    for col in requested_columns
    if col in all_columns
]

missing_requested = [
    col
    for col in requested_columns
    if col not in all_columns
]

print(
    f"\nColumns selected for EDA: "
    f"{len(USECOLS)}"
)

if missing_requested:
    print("\n⚠ Columns not found in CSV:")

    for col in missing_requested:
        print(" -", col)

else:
    print("\n✓ All requested columns found.")



STEP 2 - SELECTING COLUMNS

Columns selected for EDA: 18

⚠ Columns not found in CSV:
 - site_number


Initialize Data Structures

In [21]:
print_section("STEP 3 - INITIALIZING DATA STRUCTURES")

row_count = 0
duplicate_count = 0

missing_counts = Counter()

category_counts = {
    col: Counter()
    for col in CATEGORY_COLUMNS
    if col in USECOLS
}

numeric_values = {
    col: []
    for col in NUMERIC_COLUMNS
    if col in USECOLS
}

sample_parts = []

chunk_number = 0

print("Data structures initialized.")



STEP 3 - INITIALIZING DATA STRUCTURES
Data structures initialized.


Read Data in Chunks

In [22]:
print_section("STEP 3 - READING DATA IN CHUNKS")

for chunk in pd.read_csv(
    RAW_PATH,
    usecols=USECOLS,
    chunksize=CHUNK_SIZE,
    low_memory=False
):

    chunk_number += 1

    print(
        f"Processing chunk {chunk_number} "
        f"| rows: {len(chunk):,}"
    )

    row_count += len(chunk)

    # Missing values
    for col in chunk.columns:
        missing_counts[col] += int(
            chunk[col].isna().sum()
        )

    # Duplicate rows
    duplicate_count += int(
        chunk.duplicated().sum()
    )

    # Category counts
    for col in category_counts:

        values = (
            chunk[col]
            .dropna()
            .astype(str)
            .str.strip()
        )

        category_counts[col].update(values)

    # Numeric values
    for col in numeric_values:

        values = pd.to_numeric(
            chunk[col],
            errors="coerce"
        ).dropna()

        if len(values) > 0:
            numeric_values[col].append(values)

    # Stratified sampling by parameter
    if "parameter_name" in chunk.columns:

        for parameter, group in chunk.groupby(
            "parameter_name",
            dropna=False
        ):

            n = min(
                SAMPLE_PER_PARAMETER,
                len(group)
            )

            if n > 0:
                sample_parts.append(
                    group.sample(
                        n=n,
                        random_state=RANDOM_STATE
                    )
                )

print("\n✓ Finished reading CSV")



STEP 3 - READING DATA IN CHUNKS
Processing chunk 1 | rows: 100,000
Processing chunk 2 | rows: 100,000
Processing chunk 3 | rows: 100,000
Processing chunk 4 | rows: 100,000
Processing chunk 5 | rows: 100,000
Processing chunk 6 | rows: 100,000
Processing chunk 7 | rows: 100,000
Processing chunk 8 | rows: 100,000
Processing chunk 9 | rows: 100,000
Processing chunk 10 | rows: 100,000
Processing chunk 11 | rows: 100,000
Processing chunk 12 | rows: 100,000
Processing chunk 13 | rows: 100,000
Processing chunk 14 | rows: 100,000
Processing chunk 15 | rows: 100,000
Processing chunk 16 | rows: 100,000
Processing chunk 17 | rows: 100,000
Processing chunk 18 | rows: 100,000
Processing chunk 19 | rows: 100,000
Processing chunk 20 | rows: 100,000


KeyboardInterrupt: 

Basic Dataset Summary

In [ ]:
print_section("STEP 4 - DATASET SUMMARY")

print(f"\nTotal rows: {row_count:,}")

print(
    f"Total columns used: {len(USECOLS)}"
)

print(
    f"Duplicate rows detected inside chunks: "
    f"{duplicate_count:,}"
)

Missing Values Visualization

In [ ]:
print_section("VISUAL 1 - MISSING VALUES")

missing_plot = summary[
    summary["missing_count"] > 0
].copy()

if len(missing_plot) > 0:

    plt.figure(figsize=(12, 7))

    sns.barplot(
        data=missing_plot,
        x="missing_percent",
        y="column"
    )

    plt.title("Missing Values by Column")
    plt.xlabel("Missing Values (%)")
    plt.ylabel("Column")

    save_fig(
        OUTPUT_DIR / "missing_values.png"
    )

    print(
        "✓ Saved and displayed missing_values.png"
    )

else:

    print("\n✓ No missing values found.")

Categorical Variables

In [ ]:
print_section("STEP 5 - CATEGORICAL VARIABLES")

for col, counter in category_counts.items():

    print("\n")
    print("-" * 70)
    print(f"VARIABLE: {col}")
    print("-" * 70)

    top_values = counter.most_common(20)

    category_df = pd.DataFrame(
        top_values,
        columns=[
            col,
            "count"
        ]
    )

    display(category_df)

    safe_name = (
        col
        .replace(" ", "_")
        .replace("/", "_")
    )

    category_df.to_csv(
        OUTPUT_DIR
        / f"{safe_name}_top_values.csv",
        index=False
    )

Create Visualization Sample

In [ ]:
print_section("CREATING VISUALIZATION SAMPLE")

if sample_parts:

    sample_df = pd.concat(
        sample_parts,
        ignore_index=True
    )

    sample_df = sample_df.drop_duplicates()

    print(
        f"Visualization sample size: "
        f"{len(sample_df):,}"
    )

else:

    sample_df = pd.DataFrame()

    print("⚠ No sample data available.")